# Chest X-Ray Pneumonia Classification — Coursework

**目标**：用 ResNet-18 训练胸片肺炎二分类模型，生成论文所需的评估图表与 Grad-CAM 热力图。

**使用步骤**：
1. 右上角 **Runtime → Change runtime type → T4 GPU**
2. 运行 Step 0 确认 GPU 可用
3. 上传数据集 zip（拖入左侧文件栏）
4. 按顺序运行全部 Step
5. 最后运行 Step 11 下载结果压缩包

## Step 0: 环境检查

In [ ]:
import torch, torchvision, os, zipfile
print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision:  {torchvision.__version__}')
print(f'CUDA 可用:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号:    {torch.cuda.get_device_name(0)}')
    print(f'CUDA 版本:  {torch.version.cuda}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 显存:    {mem_gb:.1f} GB')
else:
    print('WARNING: 未检测到 GPU，请确认 Runtime 类型设为 T4 GPU')

## Step 1: 上传并解压数据集

**方式 A（推荐）**：将数据集 zip 文件拖入左侧文件栏，修改下方 `ZIP_NAME` 后运行。
**方式 B**：已上传到 Google Drive，取消注释下方代码使用。

In [ ]:
# === 方式 A: 直接上传（推荐）===
ZIP_NAME  = 'chest_xray.zip'   # ← 修改为你上传的文件名
EXTRACT_TO = '/content/dataset'

if os.path.exists(ZIP_NAME):
    os.makedirs(EXTRACT_TO, exist_ok=True)
    print(f'正在解压 {ZIP_NAME} ...')
    with zipfile.ZipFile(ZIP_NAME, 'r') as z:
        z.extractall(EXTRACT_TO)
    print('解压完成！')
    import subprocess
    result = subprocess.run(
        ['find', EXTRACT_TO, '-type', 'd', '-maxdepth', '4'],
        capture_output=True, text=True)
    print(result.stdout[:800])
else:
    print(f'未找到 {ZIP_NAME}，请先将文件拖入左侧文件栏')

## Step 2: 确认数据集路径

运行下方代码自动查找 `train/` 目录。如果找不到，手动搜索路径后设置 `DATA_ROOT`。

In [ ]:
import os

CANDIDATES = [
    '/content/dataset/chest_xray/chest_xray',
    '/content/dataset/chest_xray',
    '/content/dataset',
]

DATA_ROOT = None
for cand in CANDIDATES:
    train_path = os.path.join(cand, 'train')
    if os.path.exists(train_path):
        print(f'✓ 找到数据集: {cand}')
        DATA_ROOT = cand
        break

if DATA_ROOT is None:
    print('未自动找到 train/ 目录，请手动设置 DATA_ROOT')
else:
    for split in ['train', 'test', 'val']:
        sp = os.path.join(DATA_ROOT, split)
        if os.path.exists(sp):
            for cls in sorted(os.listdir(sp)):
                cp = os.path.join(sp, cls)
                if os.path.isdir(cp):
                    imgs = [f for f in os.listdir(cp)
                             if f.lower().endswith(('.png','.jpg','.jpeg'))]
                    print(f'  {split:5s}/{cls:12s}: {len(imgs):>5d} 张')
        else:
            print(f'  ⚠ 未找到 {split}/')

## Step 3: 安装依赖

In [ ]:
!pip install -q --upgrade pip
!pip install -q pytorch-grad-cam scikit-learn pandas seaborn
print('依赖安装完成')

## Step 4: 数据探索

In [ ]:
from PIL import Image
import numpy as np

print('=== 训练集类别分布 ===')
train_root = os.path.join(DATA_ROOT, 'train')
total = 0
for cls in sorted(os.listdir(train_root)):
    cp = os.path.join(train_root, cls)
    if os.path.isdir(cp):
        n = len([f for f in os.listdir(cp)
                 if f.lower().endswith(('.png','.jpg','.jpeg'))])
        total += n
        print(f'  {cls}: {n:>5d} 张')
print(f'  合计: {total} 张')

print('\n=== 图像尺寸采样 ===')
sample_cls = sorted(os.listdir(train_root))[0]
sample_dir = os.path.join(train_root, sample_cls)
sizes = set()
for f in sorted(os.listdir(sample_dir))[:8]:
    img = Image.open(os.path.join(sample_dir, f))
    sizes.add(img.size)
print(f'  尺寸种类: {sizes}')
print('  → 统一 Resize((224, 224))')

## Step 5: Dataset 与 DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import os

class ChestXRayDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.root = os.path.join(root, split)
        self.classes = sorted([
            d for d in os.listdir(self.root)
            if os.path.isdir(os.path.join(self.root, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for cls in self.classes:
            cls_dir = os.path.join(self.root, cls)
            for f in os.listdir(cls_dir):
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((
                        os.path.join(cls_dir, f), self.class_to_idx[cls]))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

IMG_SIZE = 224
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])
test_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

train_ds = ChestXRayDataset(DATA_ROOT, 'train', train_tf)
test_ds  = ChestXRayDataset(DATA_ROOT, 'test',  test_tf)

# val 集（如不存在则用 test 代替评估）
if os.path.exists(os.path.join(DATA_ROOT, 'val')):
    val_ds = ChestXRayDataset(DATA_ROOT, 'val', test_tf)
else:
    print('⚠ val/ 不存在，将用 test/ 做验证')
    val_ds = test_ds

print(f'Train: {len(train_ds):>5d}  Test: {len(test_ds):>5d}  Val: {len(val_ds):>5d}')
print(f'Classes: {train_ds.classes}  →  {train_ds.class_to_idx}')

In [ ]:
# 类别权重（inverse frequency，处理不平衡）
labels = torch.tensor([lb for _, lb in train_ds.samples])
class_counts = torch.bincount(labels)
class_weights = (1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * len(class_counts)
print(f'类别数量:    {class_counts.tolist()}')
print(f'类别权重:    {class_weights.tolist()}')

BATCH_SIZE = 32
train_loader = DataLoader(
    train_ds, BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True)
test_loader = DataLoader(
    test_ds, BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)
val_loader = DataLoader(
    val_ds, BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)
print(f'Batches → Train:{len(train_loader)}  Test:{len(test_loader)}  Val:{len(val_loader)}')

## Step 6: 训练 ResNet-18

- 冻结 layer1/layer2/layer3，只训练 layer4 + fc
- Early Stopping（patience=8）
- LR Scheduler（ReduceLROnPlateau, patience=4）
- 训练历史保存为 CSV，断点可恢复

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models
import time, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── 模型 ──
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for name, param in model.named_parameters():
    if any(k in name for k in ['layer1', 'layer2', 'layer3']):
        param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'总参数: {total_p:,}  |  可训练: {trainable:,}')

# ── 损失 / 优化器 / 调度器 ──
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4, verbose=True)

os.makedirs('/content/results', exist_ok=True)
HIST_FILE = '/content/results/training_history.csv'

# ── 训练参数 ──
NUM_EPOCHS  = 30
PATIENCE_ES = 8
best_val_acc = 0.0
no_improve   = 0

# 写入 CSV 表头（如文件不存在）
if not os.path.exists(HIST_FILE):
    with open(HIST_FILE, 'w') as f:
        f.write('epoch,train_loss,train_acc,val_acc,lr,time_s\n')

print(f'\n开始训练（max {NUM_EPOCHS} epochs, early_stop={PATIENCE_ES}）')
t0_all = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct += out.max(1)[1].eq(labels).sum().item()
        total += labels.size(0)

    tr_loss = run_loss / total
    tr_acc  = correct / total

    # ── Validation ──
    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            vc += model(imgs).max(1)[1].eq(labels).sum().item()
            vt += labels.size(0)
    val_acc = vc / vt if vt > 0 else 0.0

    elapsed = time.time() - t0
    curr_lr = optimizer.param_groups[0]['lr']

    # 记录历史
    with open(HIST_FILE, 'a') as f:
        f.write(f'{epoch},{tr_loss:.6f},{tr_acc:.6f},'
                f'{val_acc:.6f},{curr_lr:.2e},{elapsed:.1f}\n')

    # 调度器（基于 val_acc）
    scheduler.step(val_acc)

    # 保存最优模型
    saved = ''
    if val_acc > best_val_acc and vt > 0:
        best_val_acc = val_acc
        no_improve = 0
        torch.save(model.state_dict(), '/content/results/best_model.pth')
        saved = '  ★ saved'
    else:
        no_improve += 1

    print(f'Ep[{epoch:02d}/{NUM_EPOCHS}]  '
          f'loss={tr_loss:.4f}  tr={tr_acc:.4f}  '
          f'val={val_acc:.4f}  lr={curr_lr:.1e}  '
          f'({elapsed:.0f}s){saved}')

    # Early Stopping
    if no_improve >= PATIENCE_ES:
        print(f'\nEarly stopping (no improvement for {PATIENCE_ES} epochs)')
        break

total_time = (time.time() - t0_all) / 60
print(f'\n训练完成！用时 {total_time:.1f} min  '
      f'Best val_acc = {best_val_acc:.4f}')
print('最优模型: /content/results/best_model.pth')

## Step 7: 加载最优模型 & 读取训练历史

In [ ]:
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(
    torch.load('/content/results/best_model.pth',
               map_location=device, weights_only=True))
model.eval()
print('✓ 最优模型已加载')

hist = pd.read_csv('/content/results/training_history.csv')
print(f'训练记录: {len(hist)} epochs')
print(hist.to_string(index=False))

## Step 8: 测试集评估

计算 Accuracy、AUC、Precision、Recall、F1、Specificity，以及 per-class 详细指标（classification_report）。

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, classification_report,
)
import numpy as np

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred = out.max(1)[1].cpu().numpy()
        all_preds.extend(pred)
        all_labels.extend(labels.numpy())
        all_probs.extend(prob)

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec  = recall_score(all_labels, all_preds)
f1   = f1_score(all_labels, all_preds)
auc  = roc_auc_score(all_labels, all_probs)
cm   = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print('=' * 50)
print(f'  Accuracy:     {acc:.4f}')
print(f'  AUC:          {auc:.4f}')
print(f'  Precision:    {prec:.4f}')
print(f'  Recall:       {rec:.4f}')
print(f'  F1-Score:     {f1:.4f}')
print(f'  Specificity:  {spec:.4f}')
print('=' * 50)
print(f'  TN={int(tn)}  FP={int(fp)}  FN={int(fn)}  TP={int(tp)}')
print()
print('Per-class 详细指标:')
print(classification_report(all_labels, all_preds,
                          target_names=test_ds.classes))

In [ ]:
# 保存指标到 CSV
import pandas as pd
metrics_row = {
    'accuracy':     acc,
    'auc':          auc,
    'precision':    prec,
    'recall':       rec,
    'f1':           f1,
    'specificity':  spec,
    'tn': int(tn), 'fp': int(fp),
    'fn': int(fn), 'tp': int(tp),
}
pd.DataFrame([metrics_row]).to_csv(
    '/content/results/metrics.csv', index=False)
print('已保存: /content/results/metrics.csv')

## Step 9: 绘制评估图表

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

CLS      = test_ds.classes
R_DIR    = '/content/results'
os.makedirs(R_DIR, exist_ok=True)

# ── 1. 混淆矩阵（原始计数 + 归一化）──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLS, yticklabels=CLS, ax=ax1,
            annot_kws={'size': 15})
ax1.set_title('Confusion Matrix (Count)', fontsize=13)
ax1.set_ylabel('True label'); ax1.set_xlabel('Predicted label')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLS, yticklabels=CLS, ax=ax2,
            vmin=0, vmax=1, annot_kws={'size': 13})
ax2.set_title('Confusion Matrix (Normalized)', fontsize=13)
ax2.set_ylabel('True label'); ax2.set_xlabel('Predicted label')

plt.tight_layout()
plt.savefig(f'{R_DIR}/confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. ROC 曲线 ──
fpr, tpr, thresh = roc_curve(all_labels, all_probs)
opt_idx = np.argmax(tpr - fpr)
opt_thresh = thresh[opt_idx]

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#2980b9', lw=2, label=f'AUC = {auc:.4f}')
plt.fill_between(fpr, tpr, alpha=0.15, color='#2980b9')
plt.plot([0, 1], [0, 1], '--', color='gray', lw=1)
plt.scatter([fpr[opt_idx]], [tpr[opt_idx]],
            color='red', s=60, zorder=5,
            label=f'Best threshold = {opt_thresh:.3f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{R_DIR}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. 训练曲线 ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(hist['train_loss'], color='#e74c3c', lw=2, label='Train Loss')
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True); ax1.legend()

ax2.plot(hist['train_acc'], color='#2980b9', lw=2, label='Train Acc')
if hist['val_acc'].max() > 0:
    ax2.plot(hist['val_acc'],  color='#27ae60', lw=2, label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.grid(True); ax2.legend()

plt.suptitle('Training History', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{R_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'图表已保存到 {R_DIR}/')

## Step 10: Grad-CAM 热力图

对测试集随机 12 张图像生成 Grad-CAM 热力图，叠加在原图上。红色 = 模型关注区域。

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import random, os

os.makedirs('/content/results/gradcam', exist_ok=True)
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)
model.eval()

N_SAMPLES = min(12, len(test_ds))
indices = random.sample(range(len(test_ds)), N_SAMPLES)
rows = (N_SAMPLES + 3) // 4

fig, axes = plt.subplots(rows, 4, figsize=(14, 3.2 * rows))
axes = np.array(axes).flatten()

for i, idx in enumerate(indices):
    img_t, true_label = test_ds[idx]
    img_np = img_t.permute(1, 2, 0).cpu().numpy().copy()
    img_np = img_np * np.array(IMG_STD) + np.array(IMG_MEAN)
    img_np = np.clip(img_np, 0, 1)

    inp = img_t.unsqueeze(0).to(device)
    grayscale_cam = cam(
        input_tensor=inp,
        targets=[ClassifierOutputTarget(true_label)])[0, :, :]

    vis = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    pred_label = all_preds[idx]
    ok = '✓' if pred_label == true_label else '✗'
    axes[i].imshow(vis)
    axes[i].set_title(
        f'{ok} T:{test_ds.classes[true_label][:4]}  P:{test_ds.classes[int(pred_label)][:4]}',
        fontsize=10)
    axes[i].axis('off')

    from PIL import Image as PILImg
    PILImg.fromarray((vis * 255).astype(np.uint8)).save(
        f'/content/results/gradcam/sample_{idx:04d}.png')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Grad-CAM (layer4)  —  Red = High activation', fontsize=13)
plt.tight_layout()
plt.savefig('/content/results/gradcam_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Grad-CAM 已保存: /content/results/gradcam/')

## Step 11: 下载全部结果

打包 `/content/results/` 目录为 `results.zip`，在左侧文件栏右键 → 下载。

In [ ]:
import shutil, os

result_files = [
    'best_model.pth',
    'metrics.csv',
    'training_history.csv',
    'confusion_matrix.png',
    'roc_curve.png',
    'training_curves.png',
    'gradcam_overview.png',
]

print('=== results/ 内容 ===')
for fn in result_files:
    fp = f'/content/results/{fn}'
    if os.path.exists(fp):
        sz = os.path.getsize(fp) / 1024
        print(f'  ✓ {fn:28s}  ({sz:.0f} KB)')
    else:
        print(f'  ✗ {fn:28s}  (not found)')

zip_path = shutil.make_archive('/content/results', 'zip', '/content/results')
zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f'\n✓ 打包完成: {zip_path}  ({zip_sz:.1f} MB)')
print('左侧文件栏 → 右键 results.zip → 下载')